# Introducción al Aprendizaje Supervisado — Ejemplos

**Módulo 4 — Modelado y evaluación de enfoques supervisados · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`4.1_Introduccion_Aprendizaje_Supervisado.pdf`](4.1_Introduccion_Aprendizaje_Supervisado.pdf) y lleva a código lo que allí se explica.

## Contenido

1. **Regresión vs. clasificación**: la diferencia según la naturaleza de la variable objetivo.
2. **División train/test**: por qué dividir los datos y qué logra `stratify`.
3. **Métodos de validación**: K-folds, validación estratificada y Leave-One-Out.
4. **Underfitting y overfitting**: detectarlos comparando el rendimiento en entrenamiento y en prueba.
5. **Métricas de desempeño**: accuracy, precisión, recall, F1 y la curva ROC/AUC — calculadas a mano y verificadas contra `scikit-learn`.
6. **La paradoja del accuracy**: reproducimos el ejemplo exacto de las diapositivas (clasificador de spam).
7. **K-NN**: implementación desde cero, verificación contra `scikit-learn`, y el efecto de K en el sesgo/varianza del modelo.

> 💡 Este notebook **no requiere descargar nada ni tener cuenta de Kaggle**: usamos datasets sintéticos y el *Breast Cancer Wisconsin dataset*, incluido en `scikit-learn`.

In [ ]:
# Librerías que usaremos en todo el notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression, make_classification, load_breast_cancer
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold,
                                      LeaveOneOut)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_curve, roc_auc_score)
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
np.random.seed(42)

## 1. Regresión vs. clasificación

Como vimos en las diapositivas, el aprendizaje supervisado tiene dos tareas principales, según la **naturaleza de la variable objetivo**:

- **Regresión / interpolación**: la variable objetivo es **numérica**.
- **Clasificación**: la variable objetivo es **categórica**.

Generamos un ejemplo sintético de cada una, para verlas lado a lado.

In [ ]:
# Regresión: variable objetivo numérica continua
X_reg, y_reg = make_regression(n_samples=120, n_features=1, noise=15, random_state=0)

# Clasificación: variable objetivo categórica (dos clases)
X_clf, y_clf = make_classification(n_samples=120, n_features=2, n_redundant=0, n_informative=2,
                                    n_clusters_per_class=1, class_sep=1.8, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(X_reg, y_reg, s=15, alpha=0.7)
axes[0].set_title('Regresión: variable objetivo numérica')
axes[0].set_xlabel('atributo'); axes[0].set_ylabel('variable objetivo (numérica)')

axes[1].scatter(X_clf[:, 0], X_clf[:, 1], c=y_clf, cmap='bwr', s=20, alpha=0.7)
axes[1].set_title('Clasificación: variable objetivo categórica')
axes[1].set_xlabel('atributo 1'); axes[1].set_ylabel('atributo 2')

plt.tight_layout()
plt.show()

En regresión buscamos **predecir un valor numérico** (por ejemplo, un puntaje o un precio) para pares de atributos donde no lo conocemos. En clasificación buscamos **determinar a qué categoría pertenece** una observación (por ejemplo, alto/bajo riesgo, o spam/no spam).

## 2. División train/test

Para el resto del notebook usamos el **Breast Cancer Wisconsin dataset**: 569 observaciones, 30 variables numéricas derivadas de imágenes de biopsias, y una variable objetivo binaria (tumor maligno = 0, benigno = 1). Es un ejemplo real del caso de "identificación de sujetos con riesgo de enfermedad" que mencionan las diapositivas.

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

print(f'Observaciones: {X.shape[0]}, variables: {X.shape[1]}')
print('Clases:', cancer.target_names)
print('Proporción de clases en el dataset completo:', np.round(np.bincount(y) / len(y), 3))

Dividimos en entrenamiento y prueba. Usamos `stratify=y` para que la proporción de clases se mantenga igual en ambos subconjuntos — tal como explican las diapositivas para el **muestreo aleatorio estratificado** en Orange.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print('Proporción de clases en train (con stratify):', np.round(np.bincount(ytr) / len(ytr), 3))
print('Proporción de clases en test  (con stratify):', np.round(np.bincount(yte) / len(yte), 3))

# comparación: sin estratificar, la proporción puede desviarse un poco
_, _, ytr_sin, yte_sin = train_test_split(X, y, test_size=0.3, random_state=42)
print('\nProporción de clases en train (sin stratify):', np.round(np.bincount(ytr_sin) / len(ytr_sin), 3))

## 3. Métodos de validación

Las diapositivas muestran varias alternativas que ofrece Orange en el widget **Test and Score**: validación cruzada (K-folds), muestreo aleatorio, y Leave-One-Out. Veamos qué hace cada una con `scikit-learn`.

### 3.1 K-folds y validación estratificada

K-folds divide los datos en $K$ particiones mutuamente excluyentes. En cada iteración, $K-1$ particiones se usan para entrenar y la restante para evaluar. La versión **estratificada** garantiza que cada partición mantenga la misma proporción de clases que el dataset completo — igual que vimos con `stratify` en el punto anterior, pero aplicado repetidamente.

In [ ]:
# Demostración con un dataset pequeño y desbalanceado a propósito,
# para que se note la diferencia entre KFold y StratifiedKFold
n_demo = 30
X_demo = np.arange(n_demo).reshape(-1, 1)
y_demo = (X_demo[:, 0] < 15).astype(int)  # 15 de clase 1, 15 de clase 0, ordenadas

print('KFold (sin estratificar) — proporción de clase 1 en cada fold de TEST:')
for i, (train_idx, test_idx) in enumerate(KFold(n_splits=5).split(X_demo, y_demo)):
    print(f'  Fold {i+1}: {y_demo[test_idx].mean():.2f}')

print('\nStratifiedKFold — proporción de clase 1 en cada fold de TEST:')
for i, (train_idx, test_idx) in enumerate(StratifiedKFold(n_splits=5).split(X_demo, y_demo)):
    print(f'  Fold {i+1}: {y_demo[test_idx].mean():.2f}')

Con `KFold` normal, al no mezclar los datos, cada fold termina con una proporción de clases muy distinta (0, 0.5 o 1.0) porque nuestros datos de ejemplo están ordenados por clase. Con `StratifiedKFold`, **cada fold respeta la proporción 50/50** del dataset completo — es justo lo que buscamos al evaluar un modelo: que cada partición sea representativa.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
for i, (train_idx, test_idx) in enumerate(StratifiedKFold(n_splits=5).split(X_demo, y_demo)):
    colores = np.array(['#2ca02c'] * n_demo)
    colores[test_idx] = '#d62728'
    ax.scatter(range(n_demo), [i] * n_demo, c=colores, s=30, marker='s')
ax.set_yticks(range(5))
ax.set_yticklabels([f'Fold {i+1}' for i in range(5)])
ax.set_title('StratifiedKFold (K=5) — en rojo, la partición de test en cada iteración')
plt.tight_layout()
plt.show()

### 3.2 Leave-One-Out

Es el caso extremo de K-folds, donde $K$ es igual al número de muestras: en cada iteración se deja **una sola observación** para test y se entrena con el resto. Se usa especialmente cuando hay pocos datos.

In [ ]:
loo = LeaveOneOut()
n_particiones = loo.get_n_splits(X_demo)
print(f'Leave-One-Out sobre {n_demo} muestras genera {n_particiones} particiones distintas')
print('(tantas como muestras tenemos — en cada una, el conjunto de test tiene tamaño 1)')

## 4. Underfitting y overfitting

Las diapositivas explican cómo detectar estos dos problemas comparando el rendimiento en **train** y en **test**:

- **Underfitting**: rendimiento pobre en ambos conjuntos — el modelo es demasiado simple.
- **Overfitting**: rendimiento muy bueno en train pero bajo en test — el modelo "memorizó" los datos de entrenamiento y no generaliza.

Lo ilustramos variando la profundidad máxima de un árbol de decisión: un árbol muy superficial (poca profundidad) no logra capturar el patrón (underfitting); uno muy profundo termina memorizando el entrenamiento (overfitting).

In [ ]:
train_scores, test_scores = [], []
profundidades = range(1, 15)

for prof in profundidades:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=0)
    arbol.fit(Xtr, ytr)
    train_scores.append(arbol.score(Xtr, ytr))
    test_scores.append(arbol.score(Xte, yte))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(profundidades), train_scores, marker='o', label='Train')
ax.plot(list(profundidades), test_scores, marker='o', label='Test')
ax.set_xlabel('Profundidad máxima del árbol')
ax.set_ylabel('Accuracy')
ax.legend()
ax.set_title('Underfitting (izquierda) → ajuste razonable → overfitting (derecha)')
plt.tight_layout()
plt.show()

Con profundidad 1 el árbol apenas distingue las clases (underfitting leve: el train accuracy también es más bajo). A partir de profundidad 6 el árbol alcanza accuracy = 1.0 **en entrenamiento**, pero el accuracy en test ya no mejora e incluso baja un poco — el modelo empieza a memorizar particularidades del conjunto de entrenamiento que no generalizan (overfitting).

## 5. Métricas de desempeño

Entrenamos un clasificador K-NN (lo estudiaremos a fondo en la sección 7) y calculamos las métricas que mencionan las diapositivas: **accuracy, precisión, recall y F1** — primero a mano, a partir de la matriz de confusión, y luego verificamos con `scikit-learn`.

In [ ]:
# K-NN funciona por distancias, así que conviene estandarizar las variables primero
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr_s, ytr)
ypred = knn.predict(Xte_s)
yproba = knn.predict_proba(Xte_s)[:, 1]

cm = confusion_matrix(yte, ypred)
tn, fp, fn, tp = cm.ravel()
print('Matriz de confusión:')
print(pd.DataFrame(cm, index=['Real: maligno', 'Real: benigno'], columns=['Pred: maligno', 'Pred: benigno']))
print(f'\nTN={tn}  FP={fp}  FN={fn}  TP={tp}')

In [ ]:
# Las fórmulas de las diapositivas, calculadas a mano...
accuracy_manual = (tp + tn) / (tp + tn + fp + fn)
precision_manual = tp / (tp + fp)
recall_manual = tp / (tp + fn)
f1_manual = 2 * precision_manual * recall_manual / (precision_manual + recall_manual)

# ...y verificadas contra scikit-learn
print(f'Accuracy:  manual={accuracy_manual:.4f}  sklearn={accuracy_score(yte, ypred):.4f}')
print(f'Precision: manual={precision_manual:.4f}  sklearn={precision_score(yte, ypred):.4f}')
print(f'Recall:    manual={recall_manual:.4f}  sklearn={recall_score(yte, ypred):.4f}')
print(f'F1:        manual={f1_manual:.4f}  sklearn={f1_score(yte, ypred):.4f}')

assert np.isclose(accuracy_manual, accuracy_score(yte, ypred))
assert np.isclose(precision_manual, precision_score(yte, ypred))
assert np.isclose(recall_manual, recall_score(yte, ypred))
assert np.isclose(f1_manual, f1_score(yte, ypred))
print('\n✓ Las fórmulas manuales coinciden exactamente con scikit-learn.')

### Curva ROC y AUC

La curva ROC compara la **tasa de verdaderos positivos** (recall) contra la **tasa de falsos positivos**, variando el umbral de decisión. El área bajo la curva (AUC) resume qué tan bien separa el clasificador las dos clases: 1.0 es un clasificador perfecto, 0.5 equivale a adivinar al azar.

In [ ]:
fpr, tpr, umbrales = roc_curve(yte, yproba)
auc = roc_auc_score(yte, yproba)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f'K-NN (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Azar (AUC = 0.5)')
ax.set_xlabel('Tasa de falsos positivos (FPR)')
ax.set_ylabel('Tasa de verdaderos positivos (TPR)')
ax.set_title('Curva ROC')
ax.legend()
plt.tight_layout()
plt.show()

print(f'AUC = {auc:.4f}')

## 6. La paradoja del accuracy

Las diapositivas advierten que el accuracy **no siempre refleja qué tan bueno es un modelo**, especialmente cuando las clases están desbalanceadas. Reproducimos el ejemplo exacto: un clasificador de spam real, comparado con un clasificador "tonto" que etiqueta **todo** como "No spam".

In [ ]:
# Sistema real (de las diapositivas)
TP1, FN1, FP1, TN1 = 10, 15, 25, 100
accuracy_real = (TP1 + TN1) / (TP1 + FN1 + FP1 + TN1)
print(f'Sistema real:  TP={TP1} FN={FN1} FP={FP1} TN={TN1}  ->  accuracy = {accuracy_real*100:.1f}%')

# Sistema "dummy": clasifica todo como No Spam
TP2, FN2, FP2, TN2 = 0, 25, 0, 125
accuracy_dummy = (TP2 + TN2) / (TP2 + FN2 + FP2 + TN2)
print(f'Sistema dummy: TP={TP2} FN={FN2} FP={FP2} TN={TN2}  ->  accuracy = {accuracy_dummy*100:.1f}%')

assert abs(accuracy_real - 0.733333) < 1e-4
assert abs(accuracy_dummy - 0.833333) < 1e-4

El sistema **dummy**, que ni siquiera mira el contenido del correo, obtiene un accuracy **más alto** (83.3% vs. 73.3%) — pero es completamente inútil: nunca detecta un solo correo de spam (`TP=0`, `recall=0`). Esta es la **paradoja del accuracy**: en datasets desbalanceados, el accuracy puede esconder un modelo que no sirve para nada. Por eso, en el punto 5 calculamos también precisión, recall y F1 — que sí hubieran delatado al sistema dummy.

## 7. K-NN: K-Nearest Neighbors

Cerramos con el primer algoritmo clásico de las diapositivas: **K-NN**. Es "perezoso" (*lazy*): no aprende una estructura durante el entrenamiento, sino que en el momento de predecir, busca los $K$ vecinos más cercanos (según una distancia) y asigna la clase mayoritaria entre ellos.

### 7.1 Implementación desde cero

In [ ]:
def knn_desde_cero(X_train, y_train, x_nuevo, k):
    """Clasifica x_nuevo según la clase mayoritaria entre sus k vecinos más cercanos."""
    distancias = np.linalg.norm(X_train - x_nuevo, axis=1)
    idx_vecinos = np.argsort(distancias)[:k]
    etiquetas_vecinos = y_train[idx_vecinos]
    valores, conteos = np.unique(etiquetas_vecinos, return_counts=True)
    return valores[np.argmax(conteos)], idx_vecinos

# Dataset de juguete en 2D, fácil de visualizar
X2d, y2d = make_classification(n_samples=60, n_features=2, n_redundant=0, n_informative=2,
                                n_clusters_per_class=1, class_sep=1.5, random_state=7)

# Un punto nuevo deliberadamente cerca de la frontera entre clases
x_nuevo = np.array([-0.2112, 0.2510])

for k in [1, 5, 15]:
    pred_manual, idx_vecinos = knn_desde_cero(X2d, y2d, x_nuevo, k)
    pred_sklearn = KNeighborsClassifier(n_neighbors=k).fit(X2d, y2d).predict(x_nuevo.reshape(1, -1))[0]
    print(f'K={k:2d}: predicción desde cero = {pred_manual}, predicción sklearn = {pred_sklearn}')
    assert pred_manual == pred_sklearn

Como anticipan las diapositivas ("¿Y si cambio el número de vecinos?"), **la predicción cambia según K**: con K=1 y K=15 el punto nuevo queda en la clase 1, pero con K=5 cambia a la clase 0. Esto ocurre porque el punto está deliberadamente cerca de la frontera entre clases, donde el valor de K es más determinante. Visualicemos por qué.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for ax, k in zip(axes, [1, 5, 15]):
    pred, idx_vecinos = knn_desde_cero(X2d, y2d, x_nuevo, k)
    ax.scatter(X2d[:, 0], X2d[:, 1], c=y2d, cmap='bwr', s=25, alpha=0.4)
    ax.scatter(X2d[idx_vecinos, 0], X2d[idx_vecinos, 1], c=y2d[idx_vecinos], cmap='bwr',
               s=70, edgecolors='black', linewidths=1.2, label='vecinos usados')
    ax.scatter(*x_nuevo, c='gold', marker='*', s=300, edgecolors='black', label='punto nuevo')
    ax.set_title(f'K={k}  →  predicción: clase {pred}')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

### 7.2 El efecto de K en el sesgo y la varianza

Sobre el dataset real (*Breast Cancer*), comparamos el accuracy en train y en test para distintos valores de K — el mismo tipo de análisis que hicimos en la sección 4 con la profundidad del árbol.

In [ ]:
valores_k = range(1, 40, 2)
train_acc_k, test_acc_k = [], []

for k in valores_k:
    knn_k = KNeighborsClassifier(n_neighbors=k).fit(Xtr_s, ytr)
    train_acc_k.append(knn_k.score(Xtr_s, ytr))
    test_acc_k.append(knn_k.score(Xte_s, yte))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(valores_k), train_acc_k, marker='o', label='Train')
ax.plot(list(valores_k), test_acc_k, marker='o', label='Test')
ax.set_xlabel('K (número de vecinos)')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

mejor_k = list(valores_k)[int(np.argmax(test_acc_k))]
print(f'Mejor K según accuracy en test: {mejor_k} (accuracy = {max(test_acc_k):.4f})')

Con **K muy pequeño** (K=1), el modelo sigue muy de cerca al conjunto de entrenamiento (alta varianza, riesgo de sobreajuste: cada predicción depende de un solo vecino, sensible al ruido). Con **K muy grande**, el modelo promedia sobre demasiados vecinos y empieza a suavizar en exceso la frontera de decisión (alto sesgo, riesgo de *underfitting*). El mejor valor de K, como en casi todo en machine learning, está en un punto intermedio — y se elige **usando el conjunto de test (o de validación)**, nunca el de entrenamiento.

---

## Actividad abierta para discutir en clase

1. En la sección 4, ¿por qué el accuracy de **entrenamiento** nunca es un buen indicador de qué tan bien generaliza un modelo? ¿Qué conjunto deberíamos mirar siempre para tomar esa decisión?
2. En la paradoja del accuracy (sección 6), calculen la precisión y el recall del sistema dummy. ¿Qué les dicen esos dos números que el accuracy no dice?
3. En K-NN, ¿qué pasaría con el tiempo de predicción si el dataset de entrenamiento tuviera 10 millones de observaciones? (Pista: repasen la sección "lazy algorithm" de las diapositivas).
4. ¿En qué situación preferirían usar Leave-One-Out en vez de K-folds con K=5? ¿Y viceversa?